In [ ]:

import pandas as pd
from pybiomart import Dataset

# 1. Connect to the Mouse Dataset (Mus musculus)
# GRCm39 is the current standard assembly for mouse
mouse_dataset = Dataset(name='mmusculus_gene_ensembl', 
                        host='http://www.ensembl.org')

# 2. Define the attributes for Mouse
# 'ensembl_gene_id' -> The unique Ensembl ID (e.g., ENSMUSG...)
# 'external_gene_name' -> The official MGI Symbol (e.g., Cdh1)
# 'external_synonym' -> The alias/synonym (e.g., E-cad)
attributes = ['ensembl_gene_id', 'external_gene_name', 'external_synonym']

print("Fetching all Mouse gene mappings and synonyms...")

# 3. Query the data
# This returns a pandas DataFrame
df_mouse = mouse_dataset.query(attributes=attributes)

# 4. Clean and Rename for clarity
# Mouse symbols are usually Title Case (Cdh1), unlike Human (CDH1)
df_mouse.columns = ['Ensembl_Gene_ID', 'Gene name', 'Synonym']

# Remove rows where no symbol or synonym exists to keep the file size manageable
df_mouse.dropna(subset=['MGI_Symbol', 'Synonym'], how='all', inplace=True)

# 5. Export to CSV
df_mouse.to_csv('mouse_gene_synonyms.csv', index=False)

print(f"Success! Created 'mouse_gene_synonyms.csv' with {len(df_mouse)} rows.")
    

Fetching all Mouse gene mappings and synonyms...


KeyError: ['MGI_Symbol']

In [ ]:
#human data

import pandas as pd
from pybiomart import Dataset

# 1. Connect to the Human Dataset
dataset = Dataset(name='hsapiens_gene_ensembl', 
                  host='http://www.ensembl.org')

# 2. Define the attributes we want
# 'external_gene_name' is the official symbol (HGNC)
# 'external_synonym' contains the aliases like "ECAD" or "CD57"
attributes = ['ensembl_gene_id', 'external_gene_name', 'external_synonym']

# 3. Fetch the data
print("Fetching Human Gene Data...")
df_human = dataset.query(attributes=attributes)

#rename to 'ensembl_gene_id', 'symbol', 'synonym'
df_human.columns = ['Ensembl_Gene_ID', 'Gene name', 'Synonym']

# 4. Clean up the table (remove rows without a symbol)
df_human.dropna(subset=['Gene name'], inplace=True)

#save
df_human.to_csv('human_gene_synonyms.csv', index=False)

print(df_human.head())
#find CDH1
df_human[df_human['Gene name'] == 'CDH1']

Fetching Human Gene Data...
   Ensembl_Gene_ID Gene name Synonym
0  ENSG00000210049     MT-TF    MTTF
1  ENSG00000210049     MT-TF    TRNF
2  ENSG00000211459   MT-RNR1     12S
3  ENSG00000211459   MT-RNR1  MOTS-C
4  ENSG00000211459   MT-RNR1  MTRNR1


,Ensembl_Gene_ID,Gene name,Synonym
107189,ENSG00000039068,CDH1,CD324
107190,ENSG00000039068,CDH1,UVO
107191,ENSG00000039068,CDH1,UVOMORULIN


In [ ]:
# 1. Concatenate the initial dataframes
df_combined = pd.concat([df_human, df_mouse], ignore_index=True)

# 2. Get unique gene names
unique_genes = df_combined['Gene name'].unique()

# 3. Create a DataFrame for the self-synonyms all at once
df_synonyms = pd.DataFrame({
    'Ensembl_Gene_ID': None,
    'Gene name': unique_genes,
    'Synonym': unique_genes
})

# 4. Single concatenation (The "Speed Demon" move)
df_combined = pd.concat([df_combined, df_synonyms], ignore_index=True)

#make all Synonyms and gene names lowercase
df_combined['Gene name'] = df_combined['Gene name'].str.lower()
df_combined['Synonym'] = df_combined['Synonym'].str.lower()
#remove dups
print(f"Before removing duplicates: {len(df_combined)} rows")
df_combined.drop_duplicates(subset=['Gene name', 'Synonym'], inplace=True)
print(f"After removing duplicates: {len(df_combined)} rows")


Before removing duplicates: 314322 rows
After removing duplicates: 269928 rows


In [ ]:
#how many synonyms appear for multiple gene names?
synonym_counts = df_combined.groupby('Synonym')['Gene name'].nunique()
ambiguous_synonyms = synonym_counts[synonym_counts > 1]
#out of how many
total_synonyms = df_combined['Synonym'].nunique()
print(f"Number of ambiguous synonyms (linked to multiple genes): {len(ambiguous_synonyms)} out of {total_synonyms} total synonyms.")
#sort by occurrence
ambiguous_synonyms = ambiguous_synonyms.sort_values(ascending=False)
print(f"Top 10 most ambiguous synonyms (linked to multiple genes):")
print(ambiguous_synonyms.head(10))



Number of ambiguous synonyms (linked to multiple genes): 4299 out of 194524 total synonyms.
Top 10 most ambiguous synonyms (linked to multiple genes):
Synonym
trna                      22
ovalbumin                 20
alpha-1 antiproteinase    13
mt1                       12
p40                       11
hox1                      10
p14                        9
pap                        9
p18                        9
hox2                       9
Name: Gene name, dtype: int64


In [ ]:
#find CD57
#define mapping from Synonym to Gene name
synonym_to_gene = df_combined.groupby('Synonym')['Gene name'].apply(set).to_dict()
#look up CD57
cd57_genes = synonym_to_gene.get('hox1', set())

{'hoxa1',
 'hoxa10',
 'hoxa11',
 'hoxa13',
 'hoxa3',
 'hoxa4',
 'hoxa5',
 'hoxa6',
 'hoxa7',
 'hoxa9'}

In [ ]:
from rapidfuzz.distance import Levenshtein

def closest_string_ultra_fast(strings, custom_string):
    words = custom_string.split()
    variants = [custom_string]
    if len(words) == 3:
        variants.extend([
            f"{words[0]} {words[2]}",
            f"{words[1]} {words[2]}",
            f"{words[0]} {words[1]}"
        ])
    
    variant_data = []
    for v in variants:
        variant_data.append({
            'text': v,
            'set': set(v),
            'sorted': " ".join(sorted(v.split()))
        })

    closest = None
    # FIX 1: Use a large integer instead of float('inf')
    min_dist = 999 
    
    for original_s in strings:
        s_set = set(original_s)
        
        # Pre-filter: Character overlap
        if len(s_set.intersection(variant_data[0]['set'])) < 3:
            continue
            
        # Pre-filter: Length difference
        if abs(len(original_s) - len(custom_string)) >= min_dist:
            continue

        sorted_s = " ".join(sorted(original_s.split()))
        
        for data in variant_data:
            # FIX 2: Ensure score_cutoff is an integer and not negative
            # RapidFuzz uses score_cutoff as "if distance > cutoff, stop"
            cutoff = int(min_dist - 1)
            if cutoff < 0: 
                # If we already found a perfect match (0), we can't do better
                return closest 

            d = Levenshtein.distance(sorted_s, data['sorted'], score_cutoff=cutoff)
            
            if d < min_dist:
                min_dist = d
                closest = original_s
                
    return closest

In [ ]:
#define function that takes input, makes it lowercase, and then finds the Gene name returned. If nothing, it uses levehstein with customly set distance, to find closest synonym, then map back to gene name
def find_gene_name(input_string, synonym_to_gene, max_distance=1):
    input_string = input_string.lower()
    
    # First try exact match
    if input_string in synonym_to_gene:
        return synonym_to_gene[input_string]
    
    # If no exact match, find closest synonym
    closest_synonym = closest_string_ultra_fast(synonym_to_gene.keys(), input_string)
    
    # Check if the closest synonym is within the max distance threshold
    if Levenshtein.distance(closest_synonym, input_string) <= max_distance:
        return synonym_to_gene[closest_synonym]
    
    return None  # No suitable match found


find_gene_name("pdcd1", synonym_to_gene)

{'mki67'}